# SBO特征值校验脚本

## 功能说明
校验CSV文件中特征值与接口返回值的一致性
- 使用cust_no和use_create_time作为主键
- 从E列开始对比特征值
- 支持多线程并发请求
- 生成详细的分析报告

## 使用方法
按顺序执行各个代码块即可完成特征值校验流程

## 导入库和全局设置

In [ ]:
# 导入必要的库
import csv
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any, Dict, List, Optional

import requests

print("✓ 所有库导入成功")

## FeatureChecker类定义 - 初始化和工具方法

In [ ]:
class FeatureChecker:
    """特征值校验器"""

    def __init__(
        self,
        api_url: str,
        param1_column: int = 0,  # cust_no所在列
        param2_column: int = 2,  # use_create_time所在列
        feature_start_column: int = 3,  # 特征开始列（E列，索引4）
        thread_count: int = 150,  # 线程数
        timeout: int = 60,
    ):
        self.api_url = api_url
        self.param1_column = param1_column
        self.param2_column = param2_column
        self.feature_start_column = feature_start_column
        self.thread_count = thread_count
        self.timeout = timeout

    def normalize_timestamp(self, time_str: str) -> str:
        """标准化时间戳格式，确保毫秒精度一致"""
        if not time_str or not time_str.strip():
            return time_str
            
        time_str = time_str.strip()
        
        # 如果已经包含T，处理现有格式
        if 'T' in time_str:
            if '.' not in time_str:
                return time_str + ".000"
            else:
                parts = time_str.split('.')
                if len(parts) == 2:
                    milliseconds = parts[1][:3].ljust(3, '0')
                    return f"{parts[0]}.{milliseconds}"
                return time_str
        else:
            # 空格分隔格式，转换为T格式
            if len(time_str) >= 19:
                base_part = time_str[0:10] + "T" + time_str[11:19]
                
                if '.' in time_str and len(time_str) > 19:
                    milliseconds = time_str[20:23].ljust(3, '0')
                    return f"{base_part}.{milliseconds}"
                else:
                    return f"{base_part}.000"
            else:
                return time_str

    def format_timestamp_for_output(self, time_str: str) -> str:
        """格式化时间戳为输出格式"""
        if not time_str or not time_str.strip():
            return time_str
        
        time_str = time_str.strip()
        
        if 'T' in time_str:
            time_str = time_str.replace('T', ' ')
        
        if '.' not in time_str:
            if len(time_str) >= 19:
                return time_str + ".000"
            else:
                return time_str
        
        parts = time_str.split('.')
        if len(parts) == 2:
            milliseconds = parts[1][:3].ljust(3, '0')
            return f"{parts[0]}.{milliseconds}"
        
        return time_str

    def format_time_for_excel(self, value: str, header: str) -> str:
        """格式化时间值以便在Excel/WPS中正确显示"""
        if not value or value == "null" or value.strip() == "":
            return value
        
        value_str = str(value).strip()
        
        header_lower = header.lower()
        is_time_field = "time" in header_lower or "date" in header_lower
        
        time_pattern = r'^\d{4}-\d{2}-\d{2}(\s+\d{2}:\d{2}:\d{2}(\.\d+)?)?'
        is_time_format = bool(re.match(time_pattern, value_str))
        
        if is_time_field or is_time_format:
            if 'T' in value_str:
                value_str = value_str.replace('T', ' ')
            
            if '.' not in value_str:
                if len(value_str) >= 19:
                    return value_str + ".000"
                else:
                    return value_str
            
            parts = value_str.split('.')
            if len(parts) == 2:
                milliseconds = parts[1][:3].ljust(3, '0')
                return f"{parts[0]}.{milliseconds}"
            
            return value_str
        
        return value_str

print("✓ FeatureChecker类基础方法定义完成")

## FeatureChecker类 - 请求和比较方法

In [ ]:
def send_request(self, cust_no: str, base_time: str) -> Optional[Dict[str, Any]]:
    """发送接口请求"""
    try:
        params = {"custNo": cust_no, "baseTime": base_time}

        response = requests.post(
            self.api_url,
            json=params,
            headers={"Content-Type": "application/json"},
            timeout=self.timeout,
        )

        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        print(f"请求失败 - cust_no={cust_no}, base_time={base_time}, 错误: {str(e)}")
        return None
    except json.JSONDecodeError as e:
        print(f"JSON解析失败 - cust_no={cust_no}, base_time={base_time}, 错误: {str(e)}")
        return None

def _get_nested_value(self, data: Dict, key: str) -> Any:
    """从嵌套的JSON中获取值"""
    keys = key.split(".")
    current = data

    for k in keys:
        if isinstance(current, dict) and k in current:
            current = current[k]
        else:
            return None

    return current

def compare_values(self, csv_value: Any, api_value: Any, header: str = "") -> bool:
    """比较CSV值和接口值是否一致"""
    # 处理空值和null值
    csv_str_check = str(csv_value).strip().lower() if csv_value is not None else ""
    csv_null = csv_value is None or csv_str_check in ["null", "none", ""]

    api_str_check = str(api_value).strip().lower() if api_value is not None else ""
    api_null = api_value is None or api_str_check in ["null", "none", ""]

    if csv_null and api_null:
        return True

    if csv_null or api_null:
        return False

    # 转换为字符串进行比较
    csv_str = str(csv_value).strip()
    api_str = str(api_value).strip()

    # 先尝试将两个值转换为数字进行比较
    try:
        csv_num = float(csv_str)
        api_num = float(api_str)
        
        if abs(csv_num - api_num) < 1e-10:
            return True
        
        if csv_num == int(csv_num) and api_num == int(api_num):
            return int(csv_num) == int(api_num)
        
        return csv_num == api_num
    except (ValueError, TypeError):
        return csv_str == api_str

# 将方法添加到FeatureChecker类
FeatureChecker.send_request = send_request
FeatureChecker._get_nested_value = _get_nested_value
FeatureChecker.compare_values = compare_values

print("✓ FeatureChecker类请求和比较方法定义完成")

## FeatureChecker类 - 数据处理方法

In [ ]:
def process_row(self, row_index: int, row: List[str], headers: List[str]) -> Dict[str, Any]:
    """处理单行数据"""
    # 检查列索引是否超出范围
    if (
        self.param1_column >= len(row)
        or self.param2_column >= len(row)
        or self.param1_column < 0
        or self.param2_column < 0
    ):
        return {
            "row_index": row_index,
            "error": f"参数列索引超出范围: param1_column={self.param1_column}, param2_column={self.param2_column}, 行长度={len(row)}",
        }

    # 获取参数值
    cust_no = row[self.param1_column].strip()
    use_create_time = row[self.param2_column].strip()
    use_credit_apply_id = row[2].strip() if len(row) > 2 else ""

    # 将use_create_time转换为标准格式
    use_create_time = use_create_time.strip()
    base_time = self.normalize_timestamp(use_create_time)

    # 发送请求
    api_response = self.send_request(cust_no, base_time)

    if api_response is None:
        return {
            "row_index": row_index,
            "error": f"接口请求失败: cust_no={cust_no}, base_time={base_time}",
        }

    # 对比特征值（从E列开始）
    comparison_results = {}
    
    for i in range(self.feature_start_column, len(headers)):
        if i >= len(row):
            break

        header = headers[i]
        # 跳过pt列
        if header.lower() == "pt":
            continue
        
        csv_value = row[i] if i < len(row) else ""

        # 获取接口值
        api_value = None
        # 1. 先在顶层查找
        if header in api_response:
            api_value = api_response[header]
        # 2. 如果找不到，尝试在data字段下查找
        elif "data" in api_response and isinstance(api_response["data"], dict):
            if header in api_response["data"]:
                api_value = api_response["data"][header]
        # 3. 如果还找不到，尝试使用点号分隔的嵌套路径
        if api_value is None:
            api_value = self._get_nested_value(api_response, header)

        # 比较值
        is_match = self.compare_values(csv_value, api_value, header)

        # 保存比较结果
        comparison_results[header] = {
            "csv_value": csv_value,
            "api_value": api_value,
            "is_match": is_match,
        }

    result = {
        "row_index": row_index,
        "cust_no": cust_no,
        "use_create_time": use_create_time,
        "use_credit_apply_id": use_credit_apply_id,
        "comparison_results": comparison_results,
    }
        
    return result

# 将方法添加到FeatureChecker类
FeatureChecker.process_row = process_row

print("✓ FeatureChecker类数据处理方法定义完成")

## 配置参数设置

In [ ]:
# 配置参数
api_url = "http://10.129.2.25:8081/riskanalysis/feature/order/uncomplete"
param1_column = 1  # cust_no所在列（第1列，索引0）
param2_column = 2  # use_create_time所在列（第3列，索引2）
feature_start_column = 3  # 特征开始列（E列，索引4）
thread_count = 150  # 线程数
timeout = 60

# 文件路径设置
import os
script_dir = os.getcwd()
input_csv = os.path.join(script_dir, "qx_23_a.csv")
output_csv = input_csv.replace('.csv', '_check.csv')

print(f"API地址: {api_url}")
print(f"输入文件: {input_csv}")
print(f"输出文件: {output_csv}")
print(f"线程数: {thread_count}")
print(f"超时时间: {timeout}秒")

# 检查输入文件是否存在
if os.path.exists(input_csv):
    print(f"✓ 找到输入文件: {input_csv}")
else:
    print(f"✗ 输入文件不存在: {input_csv}")
    print("请确保CSV文件存在于当前目录")

## 创建校验器实例

In [ ]:
# 创建特征值校验器
checker = FeatureChecker(
    api_url, 
    param1_column, 
    param2_column, 
    feature_start_column, 
    thread_count, 
    timeout
)

print("✓ FeatureChecker实例创建成功")
print(f"  - API URL: {checker.api_url}")
print(f"  - 参数1列(cust_no): {checker.param1_column}")
print(f"  - 参数2列(use_create_time): {checker.param2_column}")
print(f"  - 特征开始列: {checker.feature_start_column}")
print(f"  - 线程数: {checker.thread_count}")

## 文件读取和数据准备

In [ ]:
# 读取CSV文件
def read_csv_file(file_path):
    """读取CSV文件"""
    rows = []
    headers = []
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"CSV文件不存在: {file_path}")
    
    # 尝试多种编码方式
    encodings = ["utf-8", "gbk", "gb2312", "latin-1", "cp1252", "utf-8-sig"]
    
    file_read = False
    last_error = None
    
    for encoding in encodings:
        try:
            with open(file_path, "r", encoding=encoding) as f:
                reader = csv.reader(f)
                headers = next(reader)  # 读取表头
                
                # 读取所有行
                for row in reader:
                    rows.append(row)
            
            file_read = True
            print(f"✓ 文件读取成功，使用编码: {encoding}")
            break
        except UnicodeDecodeError as e:
            last_error = e
            continue
        except Exception as e:
            last_error = e
            continue
    
    if not file_read:
        raise Exception(f"读取CSV文件失败: 尝试了多种编码方式均失败。最后错误: {str(last_error)}")
    
    return headers, rows

# 读取数据
try:
    headers, rows = read_csv_file(input_csv)
    print(f"✓ 数据读取完成")
    print(f"  - 表头数量: {len(headers)}")
    print(f"  - 数据行数: {len(rows)}")
    print(f"  - 表头预览: {headers[:5]}...")
    
    # 显示特征列信息
    feature_headers = headers[feature_start_column:]
    print(f"  - 特征列数量: {len(feature_headers)}")
    print(f"  - 特征列预览: {feature_headers[:5]}...")
    
except Exception as e:
    print(f"✗ 数据读取失败: {str(e)}")
    headers, rows = [], []

## 调试接口 - 使用第一条数据测试

In [ ]:
# 使用第一条数据调试接口
def debug_api_with_first_record(checker, headers, rows):
    """使用第一条数据调试接口，显示入参和出参"""
    
    if not rows:
        print("❌ 没有数据可用于调试")
        return None, None
    
    print("🔍 开始调试接口 - 使用第一条数据")
    print("="*80)
    
    # 获取第一条数据
    first_row = rows[0]
    
    # 检查列索引是否有效
    if (checker.param1_column >= len(first_row) or 
        checker.param2_column >= len(first_row) or
        checker.param1_column < 0 or 
        checker.param2_column < 0):
        print(f"❌ 参数列索引超出范围")
        print(f"   param1_column={checker.param1_column}, param2_column={checker.param2_column}")
        print(f"   行长度={len(first_row)}")
        return None, None
    
    # 提取参数
    cust_no = first_row[checker.param1_column].strip()
    use_create_time = first_row[checker.param2_column].strip()
    use_credit_apply_id = first_row[2].strip() if len(first_row) > 2 else ""
    
    # 标准化时间格式
    base_time = checker.normalize_timestamp(use_create_time)
    
    print("📋 第一条数据信息:")
    print(f"   行索引: 0 (第2行，包含表头)")
    print(f"   cust_no (列{checker.param1_column}): {cust_no}")
    print(f"   use_create_time (列{checker.param2_column}): {use_create_time}")
    print(f"   use_credit_apply_id (列2): {use_credit_apply_id}")
    print(f"   标准化后的base_time: {base_time}")
    
    print("\n📤 接口请求信息:")
    print(f"   URL: {checker.api_url}")
    print(f"   Method: POST")
    print(f"   Content-Type: application/json")
    print(f"   Timeout: {checker.timeout}秒")
    
    # 构建请求参数
    request_params = {
        "custNo": cust_no,
        "baseTime": base_time
    }
    
    print(f"\n📝 请求参数 (入参):")
    print(f"   {json.dumps(request_params, ensure_ascii=False, indent=4)}")
    
    print("\n🚀 发送请求...")
    
    try:
        import time
        start_time = time.time()
        
        # 发送请求
        response = requests.post(
            checker.api_url,
            json=request_params,
            headers={"Content-Type": "application/json"},
            timeout=checker.timeout,
        )
        
        elapsed_time = time.time() - start_time
        
        print(f"✅ 请求成功! 耗时: {elapsed_time:.3f}秒")
        print(f"   状态码: {response.status_code}")
        print(f"   响应大小: {len(response.content)} bytes")
        
        # 解析响应
        try:
            api_response = response.json()
            
            print(f"\n📥 接口响应 (出参):")
            print(f"   {json.dumps(api_response, ensure_ascii=False, indent=4)}")
            
            # 分析响应结构
            print(f"\n🔍 响应结构分析:")
            if isinstance(api_response, dict):
                print(f"   响应类型: 字典 (dict)")
                print(f"   顶层字段数量: {len(api_response)}")
                print(f"   顶层字段: {list(api_response.keys())}")
                
                # 检查是否有data字段
                if 'data' in api_response:
                    data_field = api_response['data']
                    if isinstance(data_field, dict):
                        print(f"   data字段类型: 字典 (dict)")
                        print(f"   data字段数量: {len(data_field)}")
                        print(f"   data字段前10个: {list(data_field.keys())[:10]}")
                    else:
                        print(f"   data字段类型: {type(data_field).__name__}")
                        print(f"   data字段值: {data_field}")
                
                # 检查特征字段匹配情况
                feature_headers = headers[checker.feature_start_column:]
                feature_headers = [h for h in feature_headers if h.lower() != 'pt']  # 排除pt列
                
                print(f"\n🎯 特征字段匹配分析:")
                print(f"   CSV特征字段数量: {len(feature_headers)}")
                
                found_in_top = 0
                found_in_data = 0
                not_found = 0
                
                sample_matches = []
                sample_not_found = []
                
                for feature in feature_headers[:20]:  # 检查前20个特征
                    if feature in api_response:
                        found_in_top += 1
                        sample_matches.append(f"{feature} (顶层)")
                    elif 'data' in api_response and isinstance(api_response['data'], dict) and feature in api_response['data']:
                        found_in_data += 1
                        sample_matches.append(f"{feature} (data字段)")
                    else:
                        not_found += 1
                        sample_not_found.append(feature)
                
                print(f"   在顶层找到: {found_in_top}")
                print(f"   在data字段找到: {found_in_data}")
                print(f"   未找到: {not_found}")
                
                if sample_matches:
                    print(f"   匹配示例: {sample_matches[:5]}")
                
                if sample_not_found:
                    print(f"   未找到示例: {sample_not_found[:5]}")
            
            else:
                print(f"   响应类型: {type(api_response).__name__}")
                print(f"   响应值: {api_response}")
            
            return request_params, api_response
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON解析失败: {str(e)}")
            print(f"   原始响应内容: {response.text[:500]}...")
            return request_params, None
            
    except requests.exceptions.Timeout:
        print(f"❌ 请求超时 (>{checker.timeout}秒)")
        return request_params, None
        
    except requests.exceptions.ConnectionError as e:
        print(f"❌ 连接错误: {str(e)}")
        return request_params, None
        
    except requests.exceptions.RequestException as e:
        print(f"❌ 请求异常: {str(e)}")
        return request_params, None
        
    except Exception as e:
        print(f"❌ 未知错误: {str(e)}")
        return request_params, None

# 执行接口调试（如果数据已准备好）
if headers and rows and checker:
    try:
        debug_request, debug_response = debug_api_with_first_record(checker, headers, rows)
        print("\n" + "="*80)
        print("✅ 接口调试完成")
        if debug_response is not None:
            print("🎉 接口调用成功，可以继续执行特征值校验")
        else:
            print("⚠️  接口调用失败，请检查网络连接和接口配置")
        print("="*80)
    except Exception as e:
        print(f"❌ 接口调试失败: {str(e)}")
        debug_request, debug_response = None, None
else:
    print("⚠️ 跳过接口调试，因为数据或校验器未准备好")
    debug_request, debug_response = None, None

## 执行特征值校验（主要处理逻辑）

In [ ]:
# 执行特征值校验的主要逻辑
def execute_feature_check(checker, headers, rows):
    """执行特征值校验"""
    
    # 准备结果数据
    results = {}
    errors = {}

    print(f"\n开始并发请求接口，线程数: {checker.thread_count}")
    start_time = time.time()

    # 使用线程池并发处理
    with ThreadPoolExecutor(max_workers=checker.thread_count) as executor:
        # 提交所有任务
        future_to_row = {
            executor.submit(checker.process_row, i, row, headers): i
            for i, row in enumerate(rows)
        }

        # 收集结果
        completed = 0
        for future in as_completed(future_to_row):
            completed += 1
            if completed % 1000 == 0:
                print(f"已完成: {completed}/{len(rows)}")

            try:
                result = future.result()
                row_index = result["row_index"]

                if "error" in result:
                    errors[row_index] = result["error"]
                else:
                    results[row_index] = result

            except Exception as e:
                row_index = future_to_row[future]
                errors[row_index] = f"处理异常: {str(e)}"

    elapsed_time = time.time() - start_time
    print(f"\n所有请求完成，耗时: {elapsed_time:.2f}秒")
    print(f"成功: {len(results)}, 失败: {len(errors)}")
    
    return results, errors

# 执行校验（如果数据已准备好）
if headers and rows:
    try:
        results, errors = execute_feature_check(checker, headers, rows)
        print("✓ 特征值校验执行完成")
    except Exception as e:
        print(f"✗ 特征值校验执行失败: {str(e)}")
        results, errors = {}, {}
else:
    print("⚠ 跳过校验执行，因为数据未准备好")
    results, errors = {}, {}

## 统计分析和结果计算

In [ ]:
# 计算统计信息
def calculate_statistics(results, headers, feature_start_column):
    """计算统计信息"""
    
    total_features = 0
    match_features = 0
    mismatch_features = 0
    
    # 统计每个特征的一致性情况
    feature_stats = {}
    for i in range(feature_start_column, len(headers)):
        header = headers[i]
        # 跳过pt列
        if header.lower() == "pt":
            continue
        feature_stats[header] = {"total": 0, "match": 0, "mismatch": 0}

    for result in results.values():
        comparison_results = result.get("comparison_results", {})
        for feature_result in comparison_results.values():
            total_features += 1
            if feature_result.get("is_match", False):
                match_features += 1
            else:
                mismatch_features += 1
        
        # 统计每个特征的一致性
        for j in range(feature_start_column, len(headers)):
            header = headers[j]
            # 跳过pt列
            if header.lower() == "pt":
                continue
            feature_result = comparison_results.get(header, {})
            is_match = feature_result.get("is_match", False)
            
            feature_stats[header]["total"] += 1
            if is_match:
                feature_stats[header]["match"] += 1
            else:
                feature_stats[header]["mismatch"] += 1
    
    # 统计无异常特征数量和有异常特征数量
    all_match_feature_count = 0
    anomaly_feature_count = 0
    anomaly_features_list = []
    
    for feature_name, stats in feature_stats.items():
        if stats["mismatch"] == 0:
            all_match_feature_count += 1
        else:
            anomaly_feature_count += 1
            match_ratio = stats["match"] / stats["total"] * 100 if stats["total"] > 0 else 0
            mismatch_ratio = stats["mismatch"] / stats["total"] * 100 if stats["total"] > 0 else 0
            anomaly_features_list.append({
                "feature_name": feature_name,
                "total": stats["total"],
                "match": stats["match"],
                "mismatch": stats["mismatch"],
                "match_ratio": match_ratio,
                "mismatch_ratio": mismatch_ratio,
            })
    
    # 按异常占比排序
    anomaly_features_list.sort(key=lambda x: x["mismatch_ratio"], reverse=True)
    
    # 计算总体匹配率
    overall_match_ratio = match_features / total_features * 100 if total_features > 0 else 0
    
    return {
        "total_features": total_features,
        "match_features": match_features,
        "mismatch_features": mismatch_features,
        "overall_match_ratio": overall_match_ratio,
        "all_match_feature_count": all_match_feature_count,
        "anomaly_feature_count": anomaly_feature_count,
        "anomaly_features_list": anomaly_features_list,
        "feature_stats": feature_stats
    }

# 计算统计信息（如果有结果数据）
if results:
    try:
        stats = calculate_statistics(results, headers, feature_start_column)
        
        print(f"\n{'='*80}")
        print(f"特征值校验结果统计")
        print(f"\n总体统计:")
        print(f"  总特征值数量: {stats['total_features']}")
        print(f"  匹配数量: {stats['match_features']}")
        print(f"  不匹配数量: {stats['mismatch_features']}")
        print(f"  总体匹配率: {stats['overall_match_ratio']:.2f}%")
        
        print(f"\n特征统计:")
        print(f"  无异常特征数量: {stats['all_match_feature_count']}")
        print(f"  有异常特征数量: {stats['anomaly_feature_count']}")
        
        if stats['anomaly_feature_count'] > 0:
            print(f"\n有异常特征详情（按异常占比降序）:")
            print(f"  {'特征名':<90} {'总数量':<10} {'异常数量':<10} {'异常占比':<10}")
            print(f"  {'-'*90} {'-'*10} {'-'*10} {'-'*10}")
            for feature in stats['anomaly_features_list'][:10]:  # 只显示前10个
                print(f"  {feature['feature_name']:<90} {feature['total']:<10} {feature['mismatch']:<10} {feature['mismatch_ratio']:.2f}%")
        
        print(f"\n{'='*90}\n")
        print("✓ 统计分析完成")
        
    except Exception as e:
        print(f"✗ 统计分析失败: {str(e)}")
        stats = {}
else:
    print("⚠ 跳过统计分析，因为没有结果数据")
    stats = {}

## 生成输出文件

In [ ]:
# 生成输出文件的函数
def write_output_files(output_csv, headers, rows, results, errors, stats):
    """写入输出文件"""
    
    base_path = output_csv.replace(".csv", "")
    
    # 定义要生成的文件路径
    api_data_path = f"{base_path}_all_api_data.csv"
    analysis_path = f"{base_path}_analysis_report.csv"
    feature_stats_path = f"{base_path}_feature_stats.csv"
    
    print(f"开始写入结果文件...")
    print(f"1. API数据文件: {api_data_path}")
    print(f"2. 分析报告文件: {analysis_path}")
    print(f"3. 特征统计文件: {feature_stats_path}")
    
    files_written = []
    
    try:
        # 1. 写入API数据文件（简化版本）
        with open(api_data_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            
            for i in range(len(rows)):
                row_data = rows[i].copy()
                
                if i in results:
                    result = results[i]
                    comparison_results = result.get("comparison_results", {})
                    
                    # 用API值替换特征值
                    for j in range(feature_start_column, len(headers)):
                        if j < len(row_data):
                            header = headers[j]
                            if header.lower() != "pt":
                                feature_result = comparison_results.get(header, {})
                                api_value = feature_result.get("api_value", None)
                                api_value_str = "null" if api_value is None else str(api_value)
                                row_data[j] = api_value_str
                
                writer.writerow(row_data)
        
        files_written.append(api_data_path)
        print(f"✅ API数据文件写入完成")
        
        # 2. 写入分析报告文件
        anomaly_records = []
        for i in range(len(rows)):
            if i in results:
                result = results[i]
                comparison_results = result.get("comparison_results", {})
                cust_no = result.get("cust_no", "")
                use_create_time = result.get("use_create_time", "")
                use_credit_apply_id = rows[i][2].strip() if len(rows[i]) > 2 else ""

                for j in range(feature_start_column, len(headers)):
                    header = headers[j]
                    if header.lower() == "pt":
                        continue
                    feature_result = comparison_results.get(header, {})
                    is_match = feature_result.get("is_match", False)

                    if not is_match:
                        csv_value = feature_result.get("csv_value", "")
                        api_value = feature_result.get("api_value", None)
                        api_value_str = "null" if api_value is None else str(api_value)
                        anomaly_records.append({
                            "feature_name": header,
                            "cust_no": cust_no,
                            "use_credit_apply_id": use_credit_apply_id,
                            "use_create_time": use_create_time,
                            "csv_value": str(csv_value),
                            "api_value": api_value_str,
                        })

        anomaly_records.sort(key=lambda x: (x["cust_no"], x["use_create_time"]))

        with open(analysis_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["特征名", "cust_no", "use_credit_apply_id", "use_create_time", "CSV值", "API值"])
            
            for record in anomaly_records:
                writer.writerow([
                    record["feature_name"],
                    record["cust_no"],
                    record["use_credit_apply_id"],
                    record["use_create_time"],
                    record["csv_value"],
                    record["api_value"],
                ])
        
        files_written.append(analysis_path)
        print(f"✅ 分析报告文件写入完成")
        
        # 3. 写入特征统计文件
        if stats:
            feature_stats = stats.get("feature_stats", {})
            all_features_list = []
            
            for feature_name, stat in feature_stats.items():
                has_anomaly = "有异常" if stat["mismatch"] > 0 else "无异常"
                match_ratio = stat["match"] / stat["total"] * 100 if stat["total"] > 0 else 0
                mismatch_ratio = stat["mismatch"] / stat["total"] * 100 if stat["total"] > 0 else 0
                
                all_features_list.append({
                    "feature_name": feature_name,
                    "has_anomaly": has_anomaly,
                    "total_count": stat["total"],
                    "match_count": stat["match"],
                    "mismatch_count": stat["mismatch"],
                    "match_ratio": match_ratio,
                    "mismatch_ratio": mismatch_ratio,
                })
            
            all_features_list.sort(key=lambda x: (x["has_anomaly"] == "无异常", -x["mismatch_count"]))
            
            no_anomaly_count = sum(1 for feature in all_features_list if feature["has_anomaly"] == "无异常")
            has_anomaly_count = sum(1 for feature in all_features_list if feature["has_anomaly"] == "有异常")
            
            with open(feature_stats_path, "w", encoding="utf-8", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["特征统计"])
                writer.writerow(["无异常特征总数", no_anomaly_count])
                writer.writerow(["有异常特征总数", has_anomaly_count])
                writer.writerow([])
                
                writer.writerow([
                    "特征名",
                    "是否有异常",
                    "比对数据条数",
                    "匹配数量",
                    "异常数量",
                    "匹配率(%)",
                    "异常率(%)"
                ])
                
                for feature in all_features_list:
                    writer.writerow([
                        feature["feature_name"],
                        feature["has_anomaly"],
                        feature["total_count"],
                        feature["match_count"],
                        feature["mismatch_count"],
                        f"{feature['match_ratio']:.2f}",
                        f"{feature['mismatch_ratio']:.2f}",
                    ])
            
            files_written.append(feature_stats_path)
            print(f"✅ 特征统计文件写入完成")
        
        return files_written
        
    except Exception as e:
        print(f"✗ 文件写入失败: {str(e)}")
        return files_written

# 生成输出文件（如果有数据）
if results and stats:
    try:
        output_files = write_output_files(output_csv, headers, rows, results, errors, stats)
        print(f"\n✓ 所有输出文件生成完成")
        for i, file_path in enumerate(output_files, 1):
            print(f"  {i}. {file_path}")
    except Exception as e:
        print(f"✗ 输出文件生成失败: {str(e)}")
else:
    print("⚠ 跳过输出文件生成，因为没有足够的数据")

## 结果总结

In [ ]:
# 显示最终结果总结
print("="*80)
print("SBO特征值校验完成！")
print("="*80)

if results and stats:
    print("执行结果:")
    print(f"  ✓ 成功处理记录数: {len(results)}")
    print(f"  ✗ 失败记录数: {len(errors)}")
    print(f"  📊 总体匹配率: {stats.get('overall_match_ratio', 0):.2f}%")
    print(f"  📈 无异常特征数: {stats.get('all_match_feature_count', 0)}")
    print(f"  📉 有异常特征数: {stats.get('anomaly_feature_count', 0)}")
    
    if 'output_files' in locals():
        print(f"\n生成的文件:")
        for i, file_path in enumerate(output_files, 1):
            if os.path.exists(file_path):
                file_size = os.path.getsize(file_path)
                print(f"  {i}. {os.path.basename(file_path)} ({file_size:,} bytes)")
            else:
                print(f"  {i}. {os.path.basename(file_path)} (文件不存在)")
else:
    print("⚠ 校验未完成或数据不足")

print("="*80)

# 如果有错误，显示部分错误信息
if errors:
    print(f"\n错误记录示例（前5个）:")
    error_items = list(errors.items())[:5]
    for row_idx, error_msg in error_items:
        print(f"  行 {row_idx + 2}: {error_msg}")

print("\n🎉 Notebook执行完成！")